In [3]:
import pandas as pd
import random

reasons = [
    "Too Expensive",
    "Not Useful",
    "Found Alternative",
    "Technical Issues",
    "Lack of Content"
]

data = []

for i in range(1000):
    data.append({
        "user_id": i,
        "subscription_months": random.randint(1,12),
        "cancel_reason": random.choice(reasons),
        "last_active_days": random.randint(1,60),
        "plan_type": random.choice(["Basic","Premium"]),
        "churn": random.choice([0,1])
    })

df = pd.DataFrame(data)

df.to_csv("users.csv",index=False)

import pandas as pd
import random

df = pd.read_csv("users.csv")

df["login_frequency"] = [
    random.randint(0,5) if churn == 1
    else random.randint(10,30)
    for churn in df["churn"]
]

df.to_csv("users.csv",index=False)

In [4]:
import sqlite3
import pandas as pd

df = pd.read_csv("users.csv")

conn = sqlite3.connect("retention.db")

df.to_sql(
    "users",
    conn,
    if_exists="replace",
    index=False
)

print("Database created")

Database created


In [10]:
query1 = """
SELECT cancel_reason,
COUNT(*) AS users
FROM users
WHERE churn = 1
GROUP BY cancel_reason
ORDER BY users DESC;
"""

print("\n TOP CHURN REASONS ")
print(pd.read_sql_query(query1, conn))



 TOP CHURN REASONS 
       cancel_reason  users
0         Not Useful    111
1    Lack of Content    104
2   Technical Issues     97
3      Too Expensive     95
4  Found Alternative     95


In [11]:
query2 = """
SELECT plan_type,
ROUND(AVG(churn) * 100, 2) AS churn_rate
FROM users
GROUP BY plan_type;
"""

print("\n CHURN RATE BY PLAN ")
print(pd.read_sql_query(query2, conn))



 CHURN RATE BY PLAN 
  plan_type  churn_rate
0     Basic       52.83
1   Premium       47.63


In [12]:
query3 = """
SELECT ROUND(AVG(subscription_months),2)
AS avg_subscription_months
FROM users;
"""

print("\n AVERAGE SUBSCRIPTION MONTHS")
print(pd.read_sql_query(query3, conn))




 AVERAGE SUBSCRIPTION MONTHS
   avg_subscription_months
0                     6.63


In [13]:
query4 = """
SELECT ROUND(AVG(subscription_months),2)
AS avg_duration
FROM users
WHERE churn = 1;
"""

print("\n AVG SUBSCRIPTION OF CHURNED USERS ")
print(pd.read_sql_query(query4, conn))



 AVG SUBSCRIPTION OF CHURNED USERS 
   avg_duration
0          6.65


In [14]:
query5 = """
SELECT
CASE
    WHEN churn = 1 THEN 'Churned'
    ELSE 'Active'
END AS status,
COUNT(*) AS users
FROM users
GROUP BY churn;
"""

print("\n ACTIVE VS CHURNED USERS ")
print(pd.read_sql_query(query5, conn))



 ACTIVE VS CHURNED USERS 
    status  users
0   Active    498
1  Churned    502


In [15]:
query6 = """
SELECT subscription_months,
COUNT(*) AS churn_count
FROM users
WHERE churn = 1
GROUP BY subscription_months
ORDER BY subscription_months;
"""

print("\n CHURN BY SUBSCRIPTION MONTH ")
print(pd.read_sql_query(query6, conn))



 CHURN BY SUBSCRIPTION MONTH 
    subscription_months  churn_count
0                     1           40
1                     2           36
2                     3           38
3                     4           36
4                     5           47
5                     6           53
6                     7           36
7                     8           31
8                     9           53
9                    10           54
10                   11           36
11                   12           42


In [16]:
query7 = """
SELECT churn,
ROUND(AVG(last_active_days),2)
AS avg_last_active
FROM users
GROUP BY churn;
"""

print("\n LAST ACTIVE DAYS ANALYSIS ")
print(pd.read_sql_query(query7, conn))



 LAST ACTIVE DAYS ANALYSIS 
   churn  avg_last_active
0      0            31.34
1      1            29.96


In [17]:
query8 = """
SELECT plan_type,
COUNT(*) AS users
FROM users
GROUP BY plan_type;
"""

print("\n PLAN DISTRIBUTION ")
print(pd.read_sql_query(query8, conn))



 PLAN DISTRIBUTION 
  plan_type  users
0     Basic    494
1   Premium    506


In [18]:
query9 = """
SELECT
cancel_reason,
ROUND(
COUNT(*) * 100.0 /
(SELECT COUNT(*) FROM users WHERE churn=1),
2
) AS percentage
FROM users
WHERE churn=1
GROUP BY cancel_reason
ORDER BY percentage DESC;
"""

print("\n CHURN PERCENTAGE BY REASON ")
print(pd.read_sql_query(query9, conn))



 CHURN PERCENTAGE BY REASON 
       cancel_reason  percentage
0         Not Useful       22.11
1    Lack of Content       20.72
2   Technical Issues       19.32
3      Too Expensive       18.92
4  Found Alternative       18.92


In [19]:
query10 = """
SELECT
COUNT(*) AS total_users,
SUM(churn) AS churned_users,
ROUND(AVG(subscription_months),2)
AS avg_subscription
FROM users;
"""

print("\n EXECUTIVE SUMMARY ")
print(pd.read_sql_query(query10, conn))

conn.close()



 EXECUTIVE SUMMARY 
   total_users  churned_users  avg_subscription
0         1000            502              6.63


In [20]:
print("\n===== ANALYSIS COMPLETE =====")
print("SQL-based churn analysis executed successfully.")
print("Data ready for AI insight generation.")


===== ANALYSIS COMPLETE =====
SQL-based churn analysis executed successfully.
Data ready for AI insight generation.
